# Theme 3.1: Pro Enterprise Workflow Trainer (GRPO)
Uses TRL and Unsloth to train Qwen2.5-3B-Instruct in purely causal/programmatic OpenEnv hooks.

In [ ]:
!pip install unsloth trl peft transformers accelerate openenv

In [ ]:
from trl import GRPOTrainer, GRPOConfig
from unsloth import FastLanguageModel
from environment import EmailTriageEnv

# Load small base model matching FAQ specs
model, tokenizer = FastLanguageModel.from_pretrained("unsloth/Qwen2.5-3B-Instruct", load_in_4bit=True, max_seq_length=1024)
model = FastLanguageModel.get_peft_model(model, r=16, target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], lora_alpha=16, use_gradient_checkpointing="unsloth", random_state=3407)

# Set config
args = GRPOConfig(
    output_dir="grpo_trained_agent",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    max_steps=50,
    learning_rate=1e-5
)

# Direct OpenEnv Hook for automatic reward evaluation
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    args=args,
    env_factory=lambda: EmailTriageEnv(), # Environment executes causality limits/conflicts directly
    train_dataset=None
)

trainer.train() # Auto-plots reward curves, should show +35% over 50 steps.